# DQN suite — the run that produces the final results

Everything in the DQN half of this thesis, in one place: five experiment grids,
resumable, with seeds as a flag rather than a constant.

| | experiment | environment | what it decides |
|---|---|---|---|
| `exp01` | ansatz / capacity | CartPole | does the circuit beat a matched classical control? |
| `exp02` | Output Reuse (block 1) | CartPole | does OR transfer off-policy? |
| `exp03` | Data Reuploading (block 2) | CartPole | does DR transfer? — the repo's clean positive |
| `exp03b` | DR with `ent=False` | CartPole | **was exp03 measuring DR, or entanglement?** |
| `exp04` | embedding / DR, stages 1–2 | FrozenLake | second environment; FIX-01 where it is measurable |

**Read this before running anything.**

*Nothing is recomputed twice.* Every cell writes a manifest and is skipped if it
exists. Interrupt whenever; rerun the same command; only what is missing runs.
That is what makes `--budget-minutes` safe and what makes the coverage seeds
count as the first seeds of a robustness pass instead of being thrown away.

*Three seeds is not a result.* `--pass coverage` (3 seeds) answers "is there an
effect worth measuring". `--pass robustness` (10) is what a conclusion needs. The
paper's own spreads — see `docs/PAPER-BASELINES.md` — put several of its
published contrasts inside one standard deviation **at 10 seeds**. Anything
written up from 3 here would be weaker than the work it critiques.

*exp03 and exp03b are one experiment in two halves.* exp03b differs by a single
boolean and exists because FIX-07 showed the `skolik` template's last entangling
ring cannot affect a PauliZ readout: depth L carries only L−1 effective
entangling blocks, so exp03's depth sweep also swept entanglement. Run one
without the other and the repo's only clean positive stays confounded.

---
## 1. Environment

In [ ]:
import os, sys, subprocess, pathlib

GITHUB_USER, REPO_NAME, BRANCH = "RogerMas99", "qrl-dissection", "main"
try:
    from google.colab import userdata
    _tok = userdata.get("GH_TOKEN")
    REPO_URL = (f"https://{_tok}@github.com/{GITHUB_USER}/{REPO_NAME}.git" if _tok
                else f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git")
except Exception:
    REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    CODE    = pathlib.Path("/content/qrl-dissection")
    RESULTS = pathlib.Path("/content/drive/MyDrive/tfm_qrl/results")
else:
    CODE    = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
    RESULTS = CODE / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

if IN_COLAB:
    if CODE.exists():
        subprocess.run(["git", "-C", str(CODE), "pull", "--quiet"], check=False)
    else:
        subprocess.run(["git", "clone", "--quiet", "-b", BRANCH, REPO_URL, str(CODE)], check=True)
    # SimplyQRL is vendored in this repo, so `pip install -e .` is the whole
    # install. No git dependency to resolve, which is what FIX-04 was about.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(CODE)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "jax", "jaxlib"], check=False)

sys.path.insert(0, str(CODE / "src"))
rev = subprocess.run(["git", "-C", str(CODE), "rev-parse", "--short", "HEAD"],
                     capture_output=True, text=True).stdout.strip()
print(f"code    {CODE} @ {rev}")
print(f"results {RESULTS}")

_need = False
if "autoray.autoray" in sys.modules:
    import autoray.autoray as _aa; _need = not hasattr(_aa, "NumpyMimic")
if "jax" in sys.modules: _need = True
if _need:
    print("\nIncompatible modules already loaded -> restarting runtime."); os.kill(os.getpid(), 9)
else:
    print("\nEnvironment clean. Continue below.")

---
## 2. Preflight

Tests, vendored-library integrity, and the FIX-05 gate — before a single cell.

FIX-05 is the reason this is not ceremony. On a `Discrete` observation space the
replay path fails *silently*: the built-in transformers return one sample's
angles for the whole batch, so a FrozenLake run completes and produces a
plausible flat curve trained on one observation repeated 128 times. There is
nothing in the output that would tell you.

In [ ]:
!cd {CODE} && python scripts/run_dqn_suite.py --plan --pass coverage --outroot {RESULTS}

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"],
                   cwd=str(CODE), capture_output=True, text=True)
print(r.stdout[-2500:])
assert r.returncode == 0, "test suite failing - do not run the grid against it"
print("\npreflight OK")

---
## 2b. What do I already have?

Worth running before anything else, because previous sessions left results and
you should know their shape before adding to them.

**Where the files are, since this trips everyone up once.** `drive.mount()` makes
your Drive appear as an ordinary folder inside Colab. After mounting,
`/content/drive/MyDrive/tfm_qrl/exp03` **is** the Drive folder — writing there is
writing to Drive. There is no separate upload step, and the code never copies
anything anywhere.

The distinction that matters:

| path | survives the runtime dying? |
|---|---|
| `/content/anything` | **no** — wiped when Colab recycles |
| `/content/drive/MyDrive/...` | yes — it is Drive |
| your laptop | only if you download it from drive.google.com |

Every notebook in this repo writes to the Drive path. So your exp01–exp03 runs,
including the full episode CSVs, are sitting in Drive right now.

In [ ]:
!cd {CODE} && python scripts/inventory_results.py /content/drive/MyDrive/tfm_qrl -v

### If it says *legacy manifests*

Those runs are fine. Early scripts simply did not record the step budget or the
DQN hyper-parameters, so the reuse guard cannot verify them and would recompute
them. Back-fill once and they count toward the robustness pass instead.

`--dqn-kwargs` cannot be recovered from any file — take it from the script that
produced the runs, not from memory, because whatever you write becomes the value
every future run is checked against. exp03 used the values below; exp02's are in
its own script.

Run `--dry-run` first; it writes nothing.

In [ ]:
!cd {CODE} && python scripts/migrate_manifests.py \
    /content/drive/MyDrive/tfm_qrl/exp03 --dry-run

In [ ]:
# Remove --dry-run once the plan above looks right. Backups are written as
# *.manifest.json.bak, so this is reversible.
!cd {CODE} && python scripts/migrate_manifests.py \
    /content/drive/MyDrive/tfm_qrl/exp03 \
    --arm hybrid_fig4 \
    --dqn-kwargs '{{"batch_size":128,"buffer_size":10000,"train_frequency":10}}' \
    --dry-run

### Getting the files onto your own machine

Nothing in this repository does that, and nothing needs to: the analysis notebook
reads them from Drive directly. If you want local copies anyway — for a backup,
or to work offline — either download the `tfm_qrl` folder from
[drive.google.com](https://drive.google.com), or install Google Drive for desktop
and it will sync automatically.

What you should **not** do is point `RESULTS` at `/content/...` without the
`drive/` prefix. It will work perfectly for one session and then be gone.

---
## 3. Coverage pass — 3 seeds

Start here. It answers *is there anything to measure* for every grid, and it is
what tells you the wall-clock cost of the robustness pass on this machine.

Measured throughput on a CPU Colab runtime: classical arms ~550 steps/s, 1-qubit
hybrid ~50, 4-qubit ~25, 8-qubit slower. Expect hours, not minutes, and use
`--budget-minutes` to fit a session — it stops after the current cell and loses
nothing.

In [ ]:
!cd {CODE} && python scripts/run_dqn_suite.py \
    --pass coverage --outroot {RESULTS} --budget-minutes 180 --skip-preflight

### Resuming

Rerun the cell above. Finished cells are skipped, so each session picks up where
the last stopped. Check progress at any time without running anything:

In [ ]:
!cd {CODE} && python scripts/run_dqn_suite.py --plan --pass coverage --outroot {RESULTS}

---
## 4. Targeted runs

While the coverage pass is incomplete, this is how to prioritise. The pair below
is the highest-value thing in the suite: it decides what exp03 actually measured.

In [ ]:
!cd {CODE} && python scripts/run_dqn_suite.py \
    --only exp03 exp03b --pass coverage --outroot {RESULTS} --skip-preflight

FrozenLake, stage 1 then stage 2. Stage 1 is classical and cheap, and carries the
liveness gate: if `frozen_onehot_mlp` does not reach a success rate near 1.0, the
regime is dead and no stage-2 number from it means anything. exp01 lost two
experiments to exactly that.

In [ ]:
!cd {CODE} && python scripts/run_dqn_suite.py \
    --only exp04 --pass coverage --outroot {RESULTS} --skip-preflight

In [ ]:
!cd {CODE} && python scripts/run_dqn_suite.py \
    --only exp04b --pass coverage --outroot {RESULTS} --skip-preflight

---
## 5. Robustness pass — 10 seeds

**This is the one that produces publishable numbers.** Seeds 1–3 already exist
from the coverage pass and are skipped, so this adds seven per cell rather than
redoing ten.

Run it in as many sessions as it takes. `--budget-minutes` bounds each session;
the manifests carry the state between them.

In [ ]:
!cd {CODE} && python scripts/run_dqn_suite.py \
    --pass robustness --outroot {RESULTS} --budget-minutes 240 --skip-preflight

In [ ]:
!cd {CODE} && python scripts/run_dqn_suite.py --plan --pass robustness --outroot {RESULTS}

---
## 6. When the grids are full

Go to `20_dqn_results.ipynb`. It reads these manifests, builds every final table
and figure, and overlays the paper's own 10-seed PPO curves from `data/`.

Then update `docs/RESULTS-LOG.md` and commit — the results log is the document
the thesis is written from, and a number that only exists in a notebook output
is a number that will be lost.